# 09 — GRU4Rec gốc và GRU4Rec đa hành vi, phiên 30 và 60 phút: chọn cấu hình trên validation

Phạm vi (protocol v1.3.2; v1.3.0 người dùng chốt ngày 24/11/2026; ngày 26/11/2026 v1.3.1 bỏ SR-GNN, v1.3.2 bỏ S-POP, SR, V-SKNN):

- **Bài toán:** chỉ dự đoán tương tác kế tiếp. Target là tập item phân biệt của nhóm timestamp kế tiếp, mọi loại hành vi, trên mọi query.
- **Hai ngưỡng phiên:** 30 phút (dữ liệu của notebook 01–04) và 60 phút (notebook này dựng lại). Hai ngưỡng dùng cùng mốc chia 0.8/0.9 của timestamp sự kiện sạch và cùng luật chia nguyên phiên; phiên cắt qua mốc thì vào `boundary_drop`.
- **Trọng tâm là hai biến thể GRU4Rec.** Hai biến thể có cùng kiến trúc, cùng siêu tham số bản chính thức (bpr-max, hidden 224, batch 80, n_sample 2048), cùng nhóm timestamp; chỉ khác đầu vào mỗi bước:
  - `gru4rec_item` (GRU4Rec gốc): trung bình embedding item của nhóm;
  - `gru4rec` (GRU4Rec đa hành vi): trung bình embedding item cộng trung bình embedding hành vi (xem / thêm giỏ / mua) của nhóm.
- **Mốc tham chiếu:** Popularity và VSTAN.

Cấu hình 30 phút của các mô hình đã chọn ở notebook 05–07 được giữ nguyên, kể cả `gru4rec` (notebook 07). Notebook này làm phần còn thiếu, đều bằng **cùng thủ tục** chọn của notebook 05–07:

| Ngưỡng | Popularity | VSTAN | `gru4rec_item` | `gru4rec` |
| --- | --- | --- | --- | --- |
| 30 phút | notebook 05 | notebook 06 | **chọn số epoch ở đây** | notebook 07 (chấm lại để kiểm tra) |
| 60 phút | **lưới trọng số ở đây** | **lưới sknn → stan → vstan ở đây** | **chọn số epoch ở đây** | **chọn số epoch ở đây** |

Số epoch được chọn bằng early stopping theo NDCG@20 trên validation (tối đa 8 epoch, patience 2). Mọi mô hình chỉ fit trên train của chính ngưỡng đó. Test không được đọc ở notebook này.

Đầu ra:
- `data/processed/session60/`;
- `models/gru4rec_item/`;
- `models/session60/{gru4rec, gru4rec_item}/`;
- `reports/09_validation_30_60/metrics.json`.

Theo dõi tiến độ: đặt biến môi trường `RR_PROGRESS_LOG` thành đường dẫn một tệp, mỗi bước sẽ ghi thêm một dòng vào tệp đó.

## 1. Cấu hình và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import itertools, os, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rr.io import report_path, environment, fingerprint, read_json, rel, require, save_verified, utc_now, verify_inputs, write_json
from rr.ablation import build_samples, known_at_for, sessionize, split_summary
from rr.candidates import CandidateFilter
from rr.evaluation import flat, run
from rr.metrics import paired_bootstrap
from rr.models import SessionKNN, popularity
from rr.samples import build_vocab, prefix_events
from rr import gru as G

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
require(PROTOCOL["version"] == "1.3.2", "Notebook này viết cho protocol v1.3.2.")
PROCESSED = ROOT / "data/processed"
DATA60 = PROCESSED / "session60"
OUT = ROOT / "reports/09_validation_30_60"
MODELS = ROOT / "models"
THRESHOLDS = {"30": PROTOCOL["locked_protocol"]["inactivity_threshold_ms"], "60": 60 * 60 * 1000}
require(sorted(THRESHOLDS.values()) == PROTOCOL["scope"]["session_thresholds_ms"], "Ngưỡng phiên khác protocol.")
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
MAX_LEN = PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1
SEED = PROTOCOL["seed"]
PAIRED = PROTOCOL["evaluation"]["paired_bootstrap"]
WIDTH = 200
GRU_MAX_EPOCHS, PATIENCE = 8, 2   # same procedure as notebook 07
TCOLS = ["query_id", "itemid", "target_is_repeat"]       # next-interaction task only
torch.set_num_threads(4)

LOG = os.environ.get("RR_PROGRESS_LOG")
def log(msg):
    line = f"{pd.Timestamp.now(tz='UTC'):%H:%M:%S} [09] {msg}"
    print(line, flush=True)
    if LOG:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")

prev = {n: read_json(report_path(ROOT, n)) for n in ["events_clean", "session_split", "metadata", "samples"]}
EVAL = {n: read_json(ROOT / f"reports/05_08_validation/{n}_validation.json") for n in ["popularity", "knn", "gru4rec"]}
expected = {"data/processed/events_clean.parquet": prev["events_clean"]["output"],
            "data/processed/split_config.json": prev["session_split"]["split_config"],
            "data/processed/events_sessionized.parquet": prev["session_split"]["artifacts"]["data/processed/events_sessionized.parquet"],
            "models/gru4rec/model.pt": EVAL["gru4rec"]["model_file"]}
for p in ["item_property_history", "item_catalog"]:
    expected[f"data/processed/{p}.parquet"] = prev["metadata"]["artifacts"][f"data/processed/{p}.parquet"]
for p in ["item_vocab.parquet", "samples/validation_queries.parquet", "samples/validation_targets.parquet"]:
    expected[f"data/processed/{p}"] = prev["samples"]["artifacts"][f"data/processed/{p}"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
for p in ["configs/protocol.json", "models/gru4rec/config.json",
          *[f"reports/05_08_validation/{n}_validation.json" for n in EVAL]]:
    inputs[p] = fingerprint(ROOT / p)
print("Đầu vào khớp SHA-256:", len(expected), "tệp; ghi thêm mã băm của", len(inputs) - len(expected), "tệp cấu hình")
print(environment(), "| torch", torch.__version__)
log("bắt đầu")

## 2. Ví dụ biên

- Ngưỡng 60 phút: nghỉ đúng 60 phút vẫn cùng phiên, nghỉ 61 phút thì tách phiên.
- `known_at` lấy sự kiện đầu tiên của **dữ liệu fit**, không lấy từ tập đánh giá.
- GRU4Rec gốc cho cùng trạng thái ẩn dù hành vi của nhóm khác nhau; GRU4Rec đa hành vi thì phân biệt được.
- Cấu hình GRU4Rec gốc chỉ khác cấu hình notebook 07 ở `use_behavior`.

In [ ]:
MIN = 60_000
_e = pd.DataFrame({"event_id": [0, 1, 2], "timestamp": [0, 60 * MIN, 121 * MIN], "visitorid": [1, 1, 1], "itemid": [5, 6, 7],
                   "event": pd.Categorical(["view"] * 3, categories=["view", "addtocart", "transaction"])})
assert sessionize(_e, THRESHOLDS["60"], (10**12, 2 * 10**12)).sort_values("event_id")["session_id"].tolist() == [1, 1, 2]
assert sessionize(_e, THRESHOLDS["30"], (10**12, 2 * 10**12)).sort_values("event_id")["session_id"].tolist() == [1, 2, 3]
_cat = pd.DataFrame({"itemid": [5, 6], "first_property_ms": pd.array([pd.NA, 1], dtype="Int64")})
_fit = pd.DataFrame({"itemid": [5, 6], "timestamp": [100, 50]})
assert known_at_for(_cat, _fit, "k")["k"].tolist() == [100, 1]

_view = pd.DataFrame({"session_id": [1, 1], "group_index": [1, 2], "itemid": [10, 11], "event": ["view", "view"]})
_cart = _view.assign(event=["addtocart", "view"])
_voc = build_vocab(_view)
_q = pd.DataFrame({"session_id": [1], "prefix_len_groups": [1]})
for _use in [False, True]:
    torch.manual_seed(0)
    _m = G.GRU4Rec(int(_voc["item_idx"].max()) + 1, hidden=8, constrained=True, use_behavior=_use).eval()
    _same = torch.allclose(G.query_states(_m, G.GroupData(_view, _voc), _q), G.query_states(_m, G.GroupData(_cart, _voc), _q))
    assert _same == (not _use)

GCFG30 = read_json(MODELS / "gru4rec/config.json")
ITEM_CFG = {**GCFG30, "name": GCFG30["name"] + "_item_only", "use_behavior": False}
assert {k for k in ITEM_CFG if ITEM_CFG[k] != GCFG30.get(k)} == {"name", "use_behavior"}
del _e, _cat, _fit, _view, _cart, _voc, _q, _m
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu phiên 30 phút và kiểm tra lại GRU4Rec đa hành vi của notebook 07

Chấm lại mô hình `models/gru4rec` (notebook 07) trên validation 30 phút với cùng bộ lọc ứng viên và fallback. NDCG@20 phải trùng tuyệt đối với report notebook 07. Bước này xác nhận đường đánh giá của notebook này giống notebook 07, nên số epoch chọn ở đây so được với số epoch đã chọn ở đó.

In [ ]:
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
ev30 = pd.read_parquet(PROCESSED / "events_sessionized.parquet", columns=["event_id", "session_id", "group_index", "itemid", "event", "timestamp", "split"])
ev30["split"] = ev30["split"].astype(str)
D = {"30": {"train": ev30[ev30["split"] == "train"].reset_index(drop=True), "validation": ev30[ev30["split"] == "validation"].reset_index(drop=True),
            "vocab": pd.read_parquet(PROCESSED / "item_vocab.parquet"),
            "val_q": pd.read_parquet(PROCESSED / "samples/validation_queries.parquet"),
            "val_t": pd.read_parquet(PROCESSED / "samples/validation_targets.parquet", columns=TCOLS)}}
D["30"]["flt"] = CandidateFilter.from_frames(D["30"]["vocab"], catalog, history, "known_at_train_ms")
POP_SELECTED = {"30": EVAL["popularity"]["selected"]}
D["30"]["fb"] = tuple(a[:FALLBACK_TOP_N] for a in popularity(D["30"]["train"], POP_SELECTED["30"]["fallback_weights"]))

def gru_eval(model, d):
    idx_item = G.idx_to_itemid(d["vocab"])
    items, scores = G.topn_items(model, G.query_states(model, G.GroupData(d["validation"], d["vocab"]), d["val_q"]), idx_item, WIDTH)
    return run(items, scores, d["val_q"], d["val_t"], d["flt"], *d["fb"])[:2]

t0 = time.perf_counter()
gru30 = G.build_from_config(GCFG30)
gru30.load_state_dict(torch.load(MODELS / "gru4rec/model.pt"))
rep, pq = gru_eval(gru30, D["30"])
nb07 = EVAL["gru4rec"]["runs"][GCFG30["name"]]["report"]
require(abs(rep["all"]["ndcg@20"] - nb07["all"]["ndcg@20"]) < 1e-9, f"Chấm lại GRU4Rec 30 phút: {rep['all']['ndcg@20']} khác notebook 07 {nb07['all']['ndcg@20']}.")
GRU = {"30": {"gru4rec": {"source": "notebook 07", "best_epoch": GCFG30["epochs"], "history": EVAL["gru4rec"]["runs"][GCFG30["name"]]["history"],
                          "report": rep, "config": GCFG30}}}
PER_QUERY = {("30", "gru4rec"): pq}
del gru30
log(f"GRU4Rec 30 phút của notebook 07 khớp: ndcg@20={rep['all']['ndcg@20']:.4f} ({time.perf_counter() - t0:.0f}s)")

## 4. Dựng dữ liệu phiên 60 phút

Hàm `sessionize` dùng chung cho hai ngưỡng. Trước khi dùng cho 60 phút, chạy nó với ngưỡng 30 phút: kết quả phiên, nhóm và tập của **từng sự kiện** phải trùng tuyệt đối với notebook 02.

Mẫu prefix–target, vocabulary và `known_at` dựng lại đúng luật của notebook 04. Target giữ ID gốc, kể cả item ngoài vocabulary. Cột `in_vocab` và `candidate_eligible` chỉ dùng để báo cáo, không loại khỏi mẫu số.

In [ ]:
t0 = time.perf_counter()
clean = pd.read_parquet(PROCESSED / "events_clean.parquet", columns=["event_id", "timestamp", "visitorid", "event", "itemid"])
CUTOFFS = read_json(PROCESSED / "split_config.json")["cutoff_ms"]
cols = ["event_id", "session_id", "group_index", "split"]
again30 = sessionize(clean, THRESHOLDS["30"], CUTOFFS)[cols].sort_values("event_id").reset_index(drop=True)
again30["split"] = again30["split"].astype(str)
ref30 = ev30[cols].sort_values("event_id").reset_index(drop=True)
pd.testing.assert_frame_equal(again30, ref30, check_dtype=False)
del again30, ref30
log("sessionize với 30 phút trùng notebook 02 trên từng sự kiện")

events = sessionize(clean, THRESHOLDS["60"], CUTOFFS)
del clean
SUMMARY = split_summary(events)
train_ev = events[events["split"] == "train"]
val_ev = events[events["split"] == "validation"]
trainval_ev = events[events["split"].isin(["train", "validation"])]
vocab = build_vocab(train_ev)
vocab_tv = build_vocab(trainval_ev)
known = known_at_for(catalog, train_ev, "known_at_train_ms").merge(
    known_at_for(catalog, trainval_ev, "known_at_trainval_ms")[["itemid", "known_at_trainval_ms"]], on="itemid", validate="one_to_one")
flt = CandidateFilter.from_frames(vocab, known, history, "known_at_train_ms")

samples, data_stats, next_id = {}, {}, 0
for sp in ["train", "validation", "test"]:
    q, t = build_samples(events, sp, first_query_id=next_id)
    next_id += len(q)
    tt = t.merge(q[["query_id", "query_time_ms", "target_time_ms"]], on="query_id", how="left", validate="many_to_one")
    require((tt["target_time_ms"] > tt["query_time_ms"]).all(), f"{sp}: target không nằm sau query.")
    t = t[TCOLS].copy()
    t["in_vocab"] = t["itemid"].isin(vocab["itemid"]).to_numpy()
    t["candidate_eligible"] = flt.eligible(tt["itemid"].to_numpy(), tt["query_time_ms"].to_numpy())
    flags = t.groupby("query_id").agg(n_targets=("itemid", "size"), any_in_vocab=("in_vocab", "any"),
                                      any_repeat=("target_is_repeat", "any"), all_repeat=("target_is_repeat", "all"))
    q = q.join(flags, on="query_id")
    q["n_targets"] = q["n_targets"].astype(np.int32)
    samples[sp] = (q, t)
    data_stats[sp] = {"queries": int(len(q)), "target_occurrences": int(len(t)), "oov_target_pct": 100 * float((~t["in_vocab"]).mean()),
                      "not_candidate_eligible_pct": 100 * float((~t["candidate_eligible"]).mean()),
                      "repeat_target_pct": 100 * float(t["target_is_repeat"].mean()),
                      "queries_all_oov": int((~q["any_in_vocab"]).sum())}
    require(len(q) == SUMMARY[sp]["prefix_target_pairs"], f"{sp}: số query khác số cặp prefix-target.")
require(train_ev["timestamp"].max() < samples["validation"][0]["query_time_ms"].min(), "Train phải nằm trước validation.")
require(trainval_ev["timestamp"].max() < samples["test"][0]["query_time_ms"].min(), "Train + validation phải nằm trước test.")
display(pd.DataFrame(data_stats).T)
display(pd.DataFrame(SUMMARY).T)

artifacts = {}
for frame, name, key in [(events, "events_sessionized.parquet", ["event_id"]), (vocab, "item_vocab.parquet", ["itemid"]),
                         (vocab_tv, "item_vocab_trainval.parquet", ["itemid"]),
                         (known[["itemid", "known_at_train_ms", "known_at_trainval_ms"]], "item_known_at.parquet", ["itemid"])]:
    info = save_verified(frame.reset_index(drop=True), DATA60 / name, key, ROOT); artifacts[info.pop("path")] = info
for sp, (q, t) in samples.items():
    for frame, kind, key in [(q, "queries", ["query_id"]), (t, "targets", ["query_id", "itemid"])]:
        info = save_verified(frame.reset_index(drop=True), DATA60 / f"samples/{sp}_{kind}.parquet", key, ROOT)
        artifacts[info.pop("path")] = info
val_q, val_t = samples["validation"][0], samples["validation"][1][TCOLS]
D["60"] = {"train": train_ev.reset_index(drop=True), "validation": val_ev.reset_index(drop=True), "vocab": vocab,
           "val_q": val_q, "val_t": val_t, "flt": flt}
log(f"dữ liệu 60 phút xong ({time.perf_counter() - t0:.0f}s): " + ", ".join(f"{sp} {s['queries']:,} query" for sp, s in data_stats.items()))

## 5. Popularity, phiên 60 phút (lưới trọng số như notebook 05)

In [ ]:
qids, qtimes = val_q["query_id"].to_numpy(), val_q["query_time_ms"].to_numpy()
EMPTY = np.empty((len(val_q), 0), dtype=np.int64)
grid = PROTOCOL["behavior_weights"]["grid"]
pop_reports, pop_rank = {}, {}
for v, c, b in itertools.product(grid["view"], grid["addtocart"], grid["transaction"]):
    w = dict(view=v, addtocart=c, transaction=b)
    name = f"pop_v{v}_c{c}_b{b}"
    items, scores = popularity(train_ev, w)
    report, *_ = run(EMPTY, None, val_q, val_t, flt, items[:FALLBACK_TOP_N], scores[:FALLBACK_TOP_N])
    require(report["short_lists"] == 0, f"{name}: có query thiếu item sau lọc.")
    report["weights"] = w
    pop_reports[name], pop_rank[name] = report, (items, scores)
require(len(pop_reports) == 16, "Lưới trọng số phải có 16 cấu hình như notebook 05.")
UNIFORM = "pop_v1_c1_b1"
best_pop = flat(pop_reports).sort_values("ndcg@20", ascending=False).index[0]
fallback_name = best_pop if pop_reports[best_pop]["all"]["ndcg@20"] > pop_reports[UNIFORM]["all"]["ndcg@20"] else UNIFORM
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in pop_rank[fallback_name])
D["60"]["fb"] = (fb_items, fb_scores)
POP_SELECTED["60"] = {"popularity_behavior_weights": pop_reports[best_pop]["weights"], "fallback": fallback_name,
                      "fallback_weights": pop_reports[fallback_name]["weights"]}
prefix_full = prefix_events(val_q, val_ev, ("itemid", "event"))
prefix = prefix_full[prefix_full["distance"] < MAX_GROUPS]
log(f"popularity 60 phút xong: best_pop={best_pop}, fallback={fallback_name}")

## 6. VSTAN, phiên 60 phút (đúng chuỗi chọn của notebook 06)

Chuỗi chọn gồm bốn bước:
1. Chọn cấu trúc SKNN (`k`, `sample_size`).
2. Với cấu trúc đó, chọn ba tham số STAN.
3. Chọn `lambda_idf` của VSTAN.
4. Thử các phương án trọng số hành vi.

Lần chạy VSTAN tốt nhất (NDCG@20) được giữ.

In [ ]:
UNIFORM_W = {"view": 1, "addtocart": 1, "transaction": 1}
WEIGHT_OPTIONS = {"uniform": UNIFORM_W, "w_1_2_5": {"view": 1, "addtocart": 2, "transaction": 5},
                  "w_1_5_10": {"view": 1, "addtocart": 5, "transaction": 10}, "pop_best": POP_SELECTED["60"]["popularity_behavior_weights"]}
knn_runs = {}
t_knn = time.perf_counter()

def run_sknn(name, **params):
    t0 = time.perf_counter()
    model = SessionKNN(**params).fit(train_ev)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    items, scores = model.recommend(prefix, qids, qtimes, WIDTH)
    q_s = time.perf_counter() - t0
    report, *_ = run(items, scores, val_q, val_t, flt, fb_items, fb_scores)
    require(report["short_lists"] == 0, f"{name}: có query thiếu item.")
    report.update(fit_seconds=round(fit_s, 2), query_seconds_total=round(q_s, 2))
    knn_runs[name] = report
    log(f"  {name:<42} ndcg@20={report['all']['ndcg@20']:.4f} fit={fit_s:.0f}s query={q_s:.0f}s")

DAY = 86_400_000
best_of = lambda prefix_name: max((n for n in knn_runs if n.startswith(prefix_name)), key=lambda n: knn_runs[n]["all"]["ndcg@20"])
for k in [50, 100, 200]:
    for sample in [500, 1000]:
        run_sknn(f"sknn_k{k}_s{sample}_uniform", variant="sknn", k=k, sample_size=sample)
b = best_of("sknn_")
base = {"k": int(b.split("_k")[1].split("_")[0]), "sample_size": int(b.split("_s")[1].split("_")[0])}
for l1 in [1.0, 3.0]:
    for l2 in [1, 7]:
        for l3 in [1.0, 3.0]:
            run_sknn(f"stan_k{base['k']}_s{base['sample_size']}_l1{l1}_l2{l2}d_l3{l3}_uniform", variant="stan",
                     k=base["k"], sample_size=base["sample_size"], lambda1=l1, lambda2_ms=l2 * DAY, lambda3=l3)
b = best_of("stan_")
stan = {**base, "lambda1": float(b.split("_l1")[1].split("_")[0]), "lambda2_ms": int(b.split("_l2")[1].split("d")[0]) * DAY,
        "lambda3": float(b.split("_l3")[1].split("_")[0])}
for lidf in [0.5, 1.0, 2.0]:
    run_sknn(f"vstan_k{base['k']}_s{base['sample_size']}_idf{lidf}_uniform", variant="vstan", lambda_idf=lidf, **stan)
b = best_of("vstan_")
vstan_structure = {**stan, "lambda_idf": float(b.split("_idf")[1].split("_")[0])}
for wname in [w for w in WEIGHT_OPTIONS if w != "uniform"]:
    run_sknn(f"vstan_best_{wname}", variant="vstan", weights=WEIGHT_OPTIONS[wname], **vstan_structure)
best_vstan = best_of("vstan_")
weight_name = next(w for w in WEIGHT_OPTIONS if best_vstan.endswith("_" + w))
KNN_SELECTED = {"60": {"vstan": {"structure": vstan_structure, "best_run": best_vstan, "weights": WEIGHT_OPTIONS[weight_name],
                                 "weight_name": weight_name}}}
require(len(knn_runs) == 20, f"Chuỗi chọn VSTAN phải có 20 lần chạy, có {len(knn_runs)}.")
log(f"VSTAN 60 phút xong ({(time.perf_counter() - t_knn) / 60:.0f} phút): {best_vstan}")

## 7. Chọn số epoch cho GRU4Rec

Ba lần huấn luyện được chạy theo cùng thủ tục với notebook 07: cùng hạt giống, chấm NDCG@20 trên validation sau mỗi epoch, dừng khi hai epoch liền không tốt hơn, rồi khôi phục epoch tốt nhất.

| Ngưỡng | Biến thể | Lưu tại |
| --- | --- | --- |
| 30 phút | `gru4rec_item` | `models/gru4rec_item` |
| 60 phút | `gru4rec_item` | `models/session60/gru4rec_item` |
| 60 phút | `gru4rec` | `models/session60/gru4rec` |

In [ ]:
MODEL_DIR = {"30": MODELS, "60": MODELS / "session60"}

def select_gru(th, name, base_cfg):
    d = MODEL_DIR[th] / name
    data = D[th]
    n_items = int(data["vocab"]["item_idx"].max()) + 1
    train_data = G.GroupData(data["train"], data["vocab"])
    windows = train_data.windows(MAX_LEN)
    reports, per_query = {}, {}
    def on_epoch(epoch):
        reports[epoch], per_query[epoch] = gru_eval(model, data)
        log(f"  {name} {th} phút epoch {epoch}: ndcg@20={reports[epoch]['all']['ndcg@20']:.4f}")
        return reports[epoch]["all"]["ndcg@20"]
    torch.manual_seed(SEED)
    model = G.build_from_config(base_cfg, n_items=n_items)
    t0 = time.perf_counter()
    hist = G.fit(model, train_data, windows, GRU_MAX_EPOCHS, lr=base_cfg["lr"], batch_size=base_cfg["batch_size"], n_neg=base_cfg["n_neg"],
                 seed=SEED, on_epoch=on_epoch, patience=PATIENCE, loss=base_cfg["loss"], bpreg=base_cfg["bpreg"],
                 elu=base_cfg["elu"], sample_alpha=base_cfg["sample_alpha"])
    fit_s = time.perf_counter() - t0
    best = max(reports, key=lambda e: reports[e]["all"]["ndcg@20"])
    require(abs(gru_eval(model, data)[0]["all"]["ndcg@20"] - reports[best]["all"]["ndcg@20"]) < 1e-12, "Không khôi phục được epoch tốt nhất.")
    cfg = {**base_cfg, "n_items": n_items, "epochs": best,
           "vocabulary": "train" if th == "30" else "train (60-minute sessions)"}
    d.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), d / "model.pt")
    write_json(d / "config.json", cfg)
    PER_QUERY[(th, name)] = per_query[best]
    log(f"{name} {th} phút xong ({fit_s / 60:.0f} phút): epoch tốt nhất {best}, ndcg@20={reports[best]['all']['ndcg@20']:.4f}")
    return {"source": "notebook 09", "best_epoch": best, "history": hist, "fit_seconds": round(fit_s, 1),
            "reports_by_epoch": {str(e): r for e, r in reports.items()}, "report": reports[best], "config": cfg}

GRU["30"]["gru4rec_item"] = select_gru("30", "gru4rec_item", ITEM_CFG)
GRU["60"] = {"gru4rec_item": select_gru("60", "gru4rec_item", ITEM_CFG),
             "gru4rec": select_gru("60", "gru4rec", {**GCFG30, "use_behavior": True})}

## 8. Tổng hợp trên validation và lưu báo cáo

- **Bảng 1:** cấu hình tốt nhất của từng mô hình ở mỗi ngưỡng, chấm trên validation của chính ngưỡng đó. Hai cột dùng **hai bộ query khác nhau**, nên chỉ đọc như mô tả. So sánh hai ngưỡng trên cùng bộ query nằm ở notebook 10.
- **Bảng 2:** hiệu NDCG@20 giữa GRU4Rec đa hành vi và GRU4Rec gốc. Hai mô hình được chấm trên cùng query nên dùng paired bootstrap.
- **Bảng 3:** cùng hiệu đó, tách theo target mới và target lặp.

In [ ]:
REF30 = {"popularity": EVAL["popularity"]["grid"][EVAL["popularity"]["best"]],
         "vstan": EVAL["knn"]["runs"][EVAL["knn"]["selected"]["vstan"]["best_run"]]}
BEST = {"30": {**REF30, "gru4rec_item": GRU["30"]["gru4rec_item"]["report"], "gru4rec": GRU["30"]["gru4rec"]["report"]},
        "60": {"popularity": pop_reports[best_pop], "vstan": knn_runs[best_vstan],
               "gru4rec_item": GRU["60"]["gru4rec_item"]["report"], "gru4rec": GRU["60"]["gru4rec"]["report"]}}
ORDER = ["popularity", "vstan", "gru4rec_item", "gru4rec"]
table = pd.DataFrame({(th, m): {n: BEST[th][n]["all"][m] for n in ORDER} for th in ["30", "60"] for m in ["hit@20", "mrr@20", "ndcg@20"]})
display(table.style.format("{:.4f}"))

behavior = {}
for th in ["30", "60"]:
    a, b = PER_QUERY[(th, "gru4rec_item")], PER_QUERY[(th, "gru4rec")]
    require((a["query_id"].to_numpy() == b["query_id"].to_numpy()).all(), "Hai biến thể phải cùng thứ tự query.")
    behavior[th] = paired_bootstrap(b["ndcg@20"].to_numpy(), a["ndcg@20"].to_numpy(), PAIRED["n_resamples"], PAIRED["seed"], PAIRED["level"])
display(pd.DataFrame(behavior).T[["mean_diff", "low", "high", "p_not_better"]])
parts = pd.DataFrame({(th, part): {n: BEST[th][n][part]["ndcg@20"] for n in ["gru4rec_item", "gru4rec"]}
                      for th in ["30", "60"] for part in ["all", "new_targets", "repeat_targets"]})
display(parts.style.format("{:.4f}"))

for d in ["gru4rec_item/model.pt", "gru4rec_item/config.json"]:
    artifacts[f"models/{d}"] = fingerprint(MODELS / d)
for d in ["gru4rec/model.pt", "gru4rec/config.json", "gru4rec_item/model.pt", "gru4rec_item/config.json"]:
    artifacts[f"models/session60/{d}"] = fingerprint(MODELS / "session60" / d)
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
write_json(OUT / "metrics.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
    "notebook": "notebooks/09_gru4rec_hanh_vi_phien_30_60.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "scope": PROTOCOL["scope"], "inputs": inputs, "artifacts": artifacts,
    "session_thresholds_ms": THRESHOLDS, "cutoff_ms": CUTOFFS, "sessionize_reproduces_notebook_02": True,
    "session60": {"split_summary": SUMMARY, "samples": data_stats,
                  "vocabulary": {"train": int(len(vocab)), "train_validation": int(len(vocab_tv))}},
    "selected": {"30": {"popularity": POP_SELECTED["30"], "knn": {"vstan": EVAL["knn"]["selected"]["vstan"]},
                        "gru4rec_epochs": GRU["30"]["gru4rec"]["best_epoch"], "gru4rec_item_epochs": GRU["30"]["gru4rec_item"]["best_epoch"]},
                 "60": {"popularity": POP_SELECTED["60"], "knn": KNN_SELECTED["60"],
                        "gru4rec_epochs": GRU["60"]["gru4rec"]["best_epoch"], "gru4rec_item_epochs": GRU["60"]["gru4rec_item"]["best_epoch"]}},
    "popularity60": {"grid": pop_reports, "uniform": UNIFORM, "best": best_pop},
    "vstan60": {"weight_options": WEIGHT_OPTIONS, "runs": knn_runs},
    "gru4rec": GRU,
    "validation_table": {"note": "each threshold on its own validation queries (different query sets)",
                         "rows": {th: {n: BEST[th][n]["all"] for n in ORDER} for th in ["30", "60"]}},
    "behavior_vs_item_only_paired": behavior,
    "status": "PASS"})
log("xong, báo cáo: " + rel(OUT / "metrics.json", ROOT))